# TP2 — Atención y mini-GPT character-level en español  *(versión resuelta)*

**Materia:** Aprendizaje Profundo — UNSAM  
**Tema:** Self-Attention, Transformer Decoder y modelos de lenguaje autoregresivos  
> Esta es la solución de referencia. No distribuir a los alumnos antes de la entrega.

## Objetivos

Al finalizar el TP deberías poder:

1. Implementar **scaled dot-product attention** y entender el rol de la máscara causal.
2. Implementar una capa de **Multi-Head Self-Attention**.
3. Construir un bloque Transformer decoder con **LayerNorm, atención, FFN y conexiones residuales**.
4. Entrenar un modelo autoregresivo character-level.
5. Evaluar el modelo con métricas cuantitativas y cualitativas: loss, perplexity, muestras generadas y mapas de atención.
6. Analizar críticamente qué aprendió el modelo y qué limitaciones tiene.

## Idea general

En un modelo de lenguaje autoregresivo queremos estimar

$$
P(x_1, x_2, \ldots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \ldots, x_{t-1}).
$$

Por eso, durante el entrenamiento, el modelo recibe una secuencia de entrada y aprende a predecir el próximo token.

In [ ]:
# ============================================================
# 0. Setup
# ============================================================

import math
import os
import random
from dataclasses import dataclass
from typing import Optional, Tuple, List

import numpy as np
import torch
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Device: {device}")

: 

## 1. Corpus y tokenización character-level

Si existe `martin_fierro.txt` en el mismo directorio se usa. Si no, se usa un corpus mínimo embebido en el notebook.

In [ ]:
# ============================================================
# Descarga automática del Martín Fierro (Project Gutenberg)
# Si el archivo ya existe, no se vuelve a descargar.
# ============================================================

import urllib.request

GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/14765/pg14765.txt"
LOCAL_PATH = "martin_fierro.txt"

if not os.path.exists(LOCAL_PATH):
    print("Descargando Martín Fierro desde Project Gutenberg...")
    try:
        urllib.request.urlretrieve(GUTENBERG_URL, LOCAL_PATH)
        with open(LOCAL_PATH, "r", encoding="utf-8", errors="replace") as f:
            raw = f.read()
        # Eliminar cabecera/pie de Gutenberg
        start = raw.find("MARTIN FIERRO")
        if start == -1:
            start = raw.find("Martín Fierro")
        if start == -1:
            start = 0
        end = raw.rfind("End of the Project Gutenberg")
        if end == -1:
            end = len(raw)
        clean = raw[start:end]
        with open(LOCAL_PATH, "w", encoding="utf-8") as f:
            f.write(clean)
        print(f"✅ Descargado y limpiado: {len(clean):,} caracteres guardados en '{LOCAL_PATH}'.")
    except Exception as e:
        print(f"⚠️  No se pudo descargar ({e}). Se usará el corpus mínimo incluido en el notebook.")
        if os.path.exists(LOCAL_PATH):
            os.remove(LOCAL_PATH)
else:
    with open(LOCAL_PATH, "r", encoding="utf-8", errors="replace") as f:
        _n = len(f.read())
    print(f"✅ '{LOCAL_PATH}' ya existe ({_n:,} caracteres). No se vuelve a descargar.")

In [ ]:
# ============================================================
# 1. Datos
# ============================================================

FALLBACK_TEXT = """
Aquí me pongo a cantar
al compás de la vigüela,
que al hombre que lo desvela
una pena extraordinaria,
como el ave solitaria
con el cantar se consuela.

Pido a los santos del cielo
que ayuden mi pensamiento;
les pido en este momento
que voy a cantar mi historia
me refresquen la memoria
y aclaren mi entendimiento.

Vengan santos milagrosos,
vengan todos en mi ayuda,
que la lengua se me añuda
y se me turba la vista;
pido a mi Dios que me asista
en una ocasión tan ruda.

Yo he conocido esta tierra
en que el paisano vivía
y su ranchito tenía
y sus hijos y mujer;
era una delicia el ver
cómo pasaba sus días.

Entonces, cuando el lucero
brillaba en el cielo santo,
y los gallos con su canto
nos decían que el día llegaba,
a la cocina rumbiaba
el gaucho que era un encanto.

Y sentao junto al fogón
a esperar que venga el día,
al cimarrón le prendía
hasta ponerse rechoncho,
mientras su china dormía
tapadita con su poncho.

"""

def load_corpus(path: str = "martin_fierro.txt") -> str:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        print(f"Corpus cargado desde {path}. Caracteres: {len(text):,}")
        return text
    print("No se encontró martin_fierro.txt. Usando corpus mínimo incluido en el notebook.")
    return FALLBACK_TEXT * 80

text = load_corpus()
print(text[:500])
print("\nCantidad de caracteres:", len(text))

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s: str) -> List[int]:
    return [stoi[c] for c in s if c in stoi]

def decode(ids: List[int]) -> str:
    return "".join(itos[int(i)] for i in ids)

print(f"Tamaño del vocabulario: {vocab_size}")
print("Vocabulario:", "".join(chars))

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens:   {len(val_data):,}")

## 2. Batches autoregresivos

Para cada bloque de longitud `block_size`, la entrada `x` contiene caracteres desde `t` hasta `t + block_size - 1`, y el target `y` contiene los mismos caracteres desplazados una posición hacia adelante.

```text
x = "Aquí me pong"
y = "quí me pongo"
```

In [ ]:
# ============================================================
# 2. Batches autoregresivos
# ============================================================

block_size = 128
batch_size = 8

def get_batch(split: str, batch_size: int = batch_size, block_size: int = block_size) -> Tuple[torch.Tensor, torch.Tensor]:
    source = train_data if split == "train" else val_data
    if len(source) <= block_size + 1:
        raise ValueError("El corpus es demasiado chico para el block_size elegido.")
    ix = torch.randint(0, len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in ix])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("x shape:", xb.shape)
print("y shape:", yb.shape)
print("\nEjemplo x:")
print(decode(xb[0].detach().cpu().tolist()))
print("\nEjemplo y:")
print(decode(yb[0].detach().cpu().tolist()))

## 3. Scaled Dot-Product Attention

La atención transforma tres tensores:

- **Q**: queries, lo que cada posición busca.
- **K**: keys, lo que cada posición ofrece para ser comparada.
- **V**: values, la información que se combina después de calcular los pesos.

$$
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V
$$

con `M` una máscara causal que impide mirar posiciones futuras.

### Respuesta conceptual

Un modelo autoregresivo predice $P(x_t \mid x_1, \ldots, x_{t-1})$. Si durante el entrenamiento la posición $t$ pudiera atender a posiciones futuras $t+1, t+2, \ldots$, el modelo "vería la respuesta" antes de predecirla: habría *data leakage*. La máscara causal pone $-\infty$ en los scores de posiciones futuras, de modo que tras el softmax esos pesos quedan en 0.

In [ ]:
# ============================================================
# 3. Scaled Dot-Product Attention  — SOLUCIÓN
# ============================================================

def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    causal: bool = True,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Calcula atención escalada.

    Parámetros
    ----------
    q, k, v : tensores de forma (..., T, d_k)
    causal  : si True, aplica máscara triangular inferior.

    Retorna
    -------
    out     : salida de atención, misma forma que q.
    weights : matriz de pesos de atención (..., T, T).
    """
    d_k = q.size(-1)

    # 1. scores = Q K^T / sqrt(d_k)  →  (..., T, T)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)

    # 2. Máscara causal: las posiciones futuras reciben -inf
    if causal:
        T = q.size(-2)
        # tril de unos: posición (i,j)=1 si j<=i  (pasado/presente)
        mask = torch.tril(torch.ones(T, T, device=q.device, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float("-inf"))

    # 3. Softmax sobre la dimensión de keys
    weights = F.softmax(scores, dim=-1)

    # 4. Combinación ponderada de V
    out = torch.matmul(weights, v)
    return out, weights


# Test mínimo de forma y causalidad.
B, T, C = 2, 5, 8
q = torch.randn(B, T, C)
k = torch.randn(B, T, C)
v = torch.randn(B, T, C)
out, weights = scaled_dot_product_attention(q, k, v, causal=True)

assert out.shape == (B, T, C)
assert weights.shape == (B, T, T)
# La parte triangular superior (fuera de la diagonal) debe ser 0
assert torch.allclose(weights[0].triu(1), torch.zeros_like(weights[0].triu(1)), atol=1e-6)
print("Tests básicos OK")

In [ ]:
# Visualización de la máscara causal y de una matriz de atención.
T = 12
q = torch.randn(1, T, 16)
k = torch.randn(1, T, 16)
v = torch.randn(1, T, 16)
_, w = scaled_dot_product_attention(q, k, v, causal=True)

plt.figure(figsize=(5, 4))
plt.imshow(w[0].detach().cpu())
plt.title("Pesos de atención causal")
plt.xlabel("posición atendida")
plt.ylabel("posición que consulta")
plt.colorbar()
plt.tight_layout()
plt.show()

## 4. Implementación del Transformer decoder

1. `Head` — una cabeza de atención causal.
2. `MultiHeadAttention` — varias cabezas en paralelo + proyección.
3. `FeedForward` — MLP por posición.
4. `Block` — un bloque Transformer con pre-norm y residuales.

In [ ]:
# ============================================================
# 4. Módulos del Transformer decoder  — SOLUCIÓN
# ============================================================

class Head(nn.Module):
    """Una cabeza de self-attention causal."""
    def __init__(self, n_embd: int, head_size: int, block_size: int, dropout: float):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        # Buffer no entrenable: máscara causal fija
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        B, T, C = x.shape

        # 1. Proyecciones
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)
        v = self.value(x)  # (B, T, head_size)

        # 2. Scores escalados
        head_size = k.size(-1)
        scores = q @ k.transpose(-2, -1) / math.sqrt(head_size)  # (B, T, T)

        # 3. Máscara causal (usamos el buffer registrado)
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))

        # 4. Softmax + dropout
        attn = F.softmax(scores, dim=-1)  # (B, T, T)
        attn = self.dropout(attn)

        # 5. Combinación con V
        out = attn @ v  # (B, T, head_size)

        if return_attn:
            return out, attn
        return out


class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention causal."""
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd debe ser divisible por n_head"
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([
            Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)
        ])
        self.proj    = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
        self.n_head  = n_head

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        if return_attn:
            outs, attns = [], []
            for h in self.heads:
                o, a = h(x, return_attn=True)
                outs.append(o)
                attns.append(a)          # (B, T, T) each
            # Concatenar salidas de todas las cabezas
            out = torch.cat(outs, dim=-1)               # (B, T, n_embd)
            out = self.dropout(self.proj(out))
            attn_stack = torch.stack(attns, dim=1)      # (B, n_head, T, T)
            return out, attn_stack
        else:
            out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, n_embd)
            out = self.dropout(self.proj(out))
            return out


class FeedForward(nn.Module):
    """MLP aplicada independientemente a cada posición."""
    def __init__(self, n_embd: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Block(nn.Module):
    """Bloque Transformer decoder: pre-norm, MHA con residual, FFN con residual."""
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float):
        super().__init__()
        self.sa   = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        if return_attn:
            sa_out, attn = self.sa(self.ln1(x), return_attn=True)
            x = x + sa_out                   # conexión residual
            x = x + self.ffwd(self.ln2(x))   # conexión residual
            return x, attn
        else:
            x = x + self.sa(self.ln1(x))
            x = x + self.ffwd(self.ln2(x))
            return x


print("Módulos definidos correctamente.")

## 5. MiniGPT character-level

El modelo completo suma embeddings de tokens y posicionales, pasa por varios bloques Transformer decoder y proyecta a logits sobre el vocabulario.

In [ ]:
# ============================================================
# 5. MiniGPT character-level  — SOLUCIÓN
# ============================================================

@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int = 32
    n_embd: int = 32
    n_head: int = 4
    n_layer: int = 2
    dropout: float = 0.1
    batch_size: int = 8


class MiniGPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.token_embedding_table    = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding_table = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.ModuleList([
            Block(config.n_embd, config.n_head, config.block_size, config.dropout)
            for _ in range(config.n_layer)
        ])
        self.ln_f    = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        idx: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
        return_attn: bool = False,
    ):
        B, T = idx.shape
        if T > self.config.block_size:
            raise ValueError("La longitud de contexto supera block_size")

        # 1 & 2. Embeddings de tokens y posicionales
        tok_emb = self.token_embedding_table(idx)                          # (B, T, n_embd)
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )                                                                   # (T, n_embd) → broadcast

        # 3. Suma
        x = tok_emb + pos_emb  # (B, T, n_embd)

        # 4. Bloques Transformer
        attn_maps = []
        for block in self.blocks:
            if return_attn:
                x, attn = block(x, return_attn=True)
                attn_maps.append(attn)  # (B, n_head, T, T)
            else:
                x = block(x)

        # 5. LayerNorm final + cabeza de lenguaje
        x      = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        # 6. Loss
        loss = None
        if targets is not None:
            B2, T2, V = logits.shape
            loss = F.cross_entropy(logits.view(B2 * T2, V), targets.view(B2 * T2))

        # 7. Retorno
        if return_attn:
            return logits, loss, attn_maps
        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
    ) -> torch.Tensor:
        """
        Generación autoregresiva.
        """
        for _ in range(max_new_tokens):
            # 1. Recortar contexto a block_size
            idx_cond = idx[:, -self.config.block_size:]

            # 2. Forward pass
            logits, _ = self(idx_cond)

            # 3. Logits de la última posición
            logits = logits[:, -1, :]  # (B, vocab_size)

            # 4. Temperature scaling
            logits = logits / temperature

            # 5. Top-k: mantener sólo los k logits más altos
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")

            # 6. Muestreo
            probs      = F.softmax(logits, dim=-1)
            idx_next   = torch.multinomial(probs, num_samples=1)  # (B, 1)

            # 7. Concatenar al contexto
            idx = torch.cat([idx, idx_next], dim=1)

        return idx


# ---- Instanciar y verificar ----
config = GPTConfig(
    vocab_size=vocab_size,
    block_size=128,
    n_embd=32,
    n_head=4,
    n_layer=1,
    dropout=0.1,
    batch_size=8,
)

model = MiniGPT(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nCantidad de parámetros: {n_params:,}")

xb, yb = get_batch("train", config.batch_size, config.block_size)
logits, loss = model(xb, yb)
print("logits shape:", logits.shape)
print("loss inicial:", float(loss.detach().cpu()))
print(f"loss esperado para vocab_size={vocab_size}: {math.log(vocab_size):.4f}")

## 6. Entrenamiento

La **perplexity** se define como $\operatorname{PPL} = e^{\operatorname{loss}}$. Menor PPL indica que el modelo asigna mayor probabilidad al próximo carácter correcto.

Con el corpus del Martín Fierro (~220K caracteres) y la configuración base (`block_size=128`, `n_embd=32`, `n_layer=1`, `n_head=4`) podés esperar aproximadamente:

| Iteraciones | Loss val. esperada | PPL esperada | Calidad del texto |
|---:|---:|---:|---|
| 0 | ~4.1 | ~60 | Ruido aleatorio |
| 500 | ~2.5 | ~12 | Aparecen letras frecuentes y espacios |
| 1000 | ~2.0 | ~7.4 | Palabras reconocibles, estructura de verso |
| 3000 | ~1.6 | ~5.0 | Versos con rima ocasional, palabras del corpus |
| 5000 | ~1.4 | ~4.1 | Estrofas con coherencia local |

> ⏱️ En **CPU** el entrenamiento tarda ~5–10 min para 1000 iteraciones. En **Colab con GPU** podés llegar a 5000 iteraciones en ~3 min.

### Variantes para explorar (sección 9)

Una vez que tenés el modelo base funcionando, te recomendamos probar al menos dos de estas variantes y reportar cómo cambian las métricas:

1. **Contexto más largo** — cambiá `block_size` de `128` a `256`. El modelo puede aprender dependencias más largas (estrofas completas). Observá si la val loss mejora.

2. **Más cabezas de atención** — probá `n_head=1` vs `n_head=8`. ¿Una sola cabeza es suficiente para este corpus? ¿Qué pasa con los mapas de atención?

3. **Más capas** — probá `n_layer=2` o `n_layer=4`. Más profundidad aumenta la capacidad del modelo, pero también el riesgo de overfitting con un corpus chico. Compará train vs val loss.

4. **Embedding más grande** — probá `n_embd=64` o `n_embd=128`. Más dimensiones permiten representaciones más ricas, pero también más parámetros.

5. **Dropout** — compará `dropout=0.0` vs `dropout=0.2`. ¿Ayuda la regularización con este tamaño de corpus?

6. **Learning rate** — el valor `3e-4` es un buen punto de partida, pero podés probar `1e-3` (más agresivo, converge más rápido pero puede divergir) o `1e-4` (más conservador).

> 💡 **Tip:** para comparar variantes de forma justa, entrenales con el mismo número de iteraciones y usá siempre la misma semilla (`SEED = 42`). La sección 9 ya tiene una utilidad `train_small_model` que hace esto automáticamente.

In [ ]:
# ============================================================
# 6. Entrenamiento
# ============================================================

@torch.no_grad()
def estimate_loss(model: nn.Module, eval_iters: int = 20):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(eval_iters):
            X, Y = get_batch(split, config.batch_size, config.block_size)
            _, loss = model(X, Y)
            losses.append(loss.item())
        out[split] = float(np.mean(losses))
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

max_iters     = 5000 if device != "cpu" else 1000
eval_interval = 200  if device != "cpu" else 100
history = []

for it in range(max_iters + 1):
    if it % eval_interval == 0:
        losses = estimate_loss(model, eval_iters=20)
        history.append({"iter": it, **losses})
        print(
            f"iter {it:4d} | "
            f"train loss {losses['train']:.4f} | "
            f"val loss {losses['val']:.4f} | "
            f"val ppl {math.exp(losses['val']):.2f}"
        )

    xb, yb = get_batch("train", config.batch_size, config.block_size)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

In [ ]:
iters        = [h["iter"] for h in history]
train_losses = [h["train"] for h in history]
val_losses   = [h["val"]   for h in history]

plt.figure(figsize=(6, 4))
plt.plot(iters, train_losses, marker="o", label="train")
plt.plot(iters, val_losses,   marker="o", label="val")
plt.xlabel("Iteración")
plt.ylabel("Cross-entropy loss")
plt.title("Curva de entrenamiento")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 7. Generación de texto

**`temperature`** controla la "creatividad": valores bajos (~0.5) hacen la distribución más picuda (texto más predecible), valores altos (~1.5) la aplanan (más aleatorio).  
**`top_k`** restringe el muestreo a los $k$ tokens más probables, eliminando opciones muy improbables.

In [ ]:
# ============================================================
# 7. Generación
# ============================================================

model.eval()

for temperature in [0.3, 0.5, 0.9, 1.4]:
    prompt  = "Aquí "
    context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    generated = model.generate(context, max_new_tokens=200, temperature=temperature, top_k=20)
    print(f"\n{'='*60}")
    print(f"temperature={temperature}")
    print('='*60)
    print(decode(generated[0].detach().cpu().tolist()))

model.train()

## 8. Visualización de mapas de atención

Los pesos de atención revelan a qué posiciones previas presta atención cada token. La máscara causal garantiza que la parte superior (posiciones futuras) sea exactamente 0.

In [ ]:
# ============================================================
# 8. Mapas de atención del modelo entrenado
# ============================================================

model.eval()
prompt = "Aquí me pongo"
idx    = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

with torch.no_grad():
    logits, loss, attn_maps = model(idx, return_attn=True)

labels = list(decode(idx[0].detach().cpu().tolist()))
n_layers = len(attn_maps)
n_heads  = attn_maps[0].shape[1]

fig, axes = plt.subplots(n_layers, n_heads, figsize=(4 * n_heads, 4 * n_layers))
if n_layers == 1:
    axes = [axes]
if n_heads == 1:
    axes = [[ax] for ax in axes]

for l_idx in range(n_layers):
    for h_idx in range(n_heads):
        ax   = axes[l_idx][h_idx]
        attn = attn_maps[l_idx][0, h_idx].detach().cpu()
        im = ax.imshow(attn, vmin=0, vmax=attn.max())
        ax.set_title(f"Capa {l_idx}, cabeza {h_idx}")
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=90)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels)
        plt.colorbar(im, ax=ax)

plt.suptitle("Mapas de atención", y=1.02)
plt.tight_layout()
plt.show()

model.train()

## 9. Ablation study

Comparamos varias configuraciones: `n_head=1` vs `n_head=4`, `block_size=32` vs `block_size=64`, `n_layer=1` vs `n_layer=2`.

In [ ]:
# ============================================================
# 9. Utilidad para entrenar variantes y ablation study
# ============================================================

@torch.no_grad()
def estimate_loss_for_config(m: nn.Module, cfg: GPTConfig, eval_iters: int = 10):
    m.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(eval_iters):
            X, Y = get_batch(split, cfg.batch_size, cfg.block_size)
            _, loss = m(X, Y)
            losses.append(loss.item())
        out[split] = float(np.mean(losses))
    m.train()
    return out


def train_small_model(cfg: GPTConfig, max_iters: int = 200, eval_iters: int = 10, lr: float = 3e-4):
    m   = MiniGPT(cfg).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr)
    hist = []

    for it in range(max_iters + 1):
        if it % max(1, max_iters // 4) == 0:
            losses = estimate_loss_for_config(m, cfg, eval_iters=eval_iters)
            hist.append({"iter": it, **losses})

        xb, yb = get_batch("train", cfg.batch_size, cfg.block_size)
        _, loss = m(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
        opt.step()

    final_losses = estimate_loss_for_config(m, cfg, eval_iters=eval_iters)
    return m, hist, final_losses


# Configuraciones a comparar
ablation_configs = {
    "1_head":   GPTConfig(vocab_size=vocab_size, block_size=32, n_embd=32, n_head=1, n_layer=1, dropout=0.1, batch_size=8),
    "4_heads":  GPTConfig(vocab_size=vocab_size, block_size=32, n_embd=32, n_head=4, n_layer=1, dropout=0.1, batch_size=8),
    "2_layers": GPTConfig(vocab_size=vocab_size, block_size=32, n_embd=32, n_head=4, n_layer=2, dropout=0.1, batch_size=8),
    "block64":  GPTConfig(vocab_size=vocab_size, block_size=64, n_embd=32, n_head=4, n_layer=1, dropout=0.1, batch_size=8),
}

results = []
trained_variants = {}

for name, cfg in ablation_configs.items():
    print(f"\nEntrenando: {name}")
    iters_exp = 200
    m, hist, losses = train_small_model(cfg, max_iters=iters_exp, eval_iters=5)
    n_p = sum(p.numel() for p in m.parameters())
    results.append({
        "modelo":      name,
        "params":      n_p,
        "train_loss":  round(losses["train"], 4),
        "val_loss":    round(losses["val"],   4),
        "val_ppl":     round(math.exp(losses["val"]), 2),
    })
    trained_variants[name] = m
    print(f"  params={n_p:,}  train={losses['train']:.4f}  val={losses['val']:.4f}  ppl={math.exp(losses['val']):.2f}")

print("\n" + "="*70)
print(f"{'Modelo':<12} {'Params':>8} {'Train':>10} {'Val':>10} {'PPL':>8}")
print("-"*70)
for r in results:
    print(f"{r['modelo']:<12} {r['params']:>8,} {r['train_loss']:>10.4f} {r['val_loss']:>10.4f} {r['val_ppl']:>8.2f}")

In [ ]:
# Generación de cada variante
prompt  = "Aquí "
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

for name, m in trained_variants.items():
    m.eval()
    gen = m.generate(context, max_new_tokens=150, temperature=0.9, top_k=20)
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(decode(gen[0].detach().cpu().tolist()))
    m.train()

## 10. Informe dentro del notebook

### 10.1 Implementación — descripción de cada módulo

| Módulo | Descripción |
|---|---|
| `scaled_dot_product_attention` | Calcula $\text{softmax}(QK^T/\sqrt{d_k}+M)V$. La máscara $M$ pone $-\infty$ sobre la diagonal superior para impedir que el modelo vea tokens futuros. |
| `Head` | Proyecta el embedding a espacios Q, K, V de dimensión `head_size`, aplica la atención escalada causal con máscara y devuelve la combinación ponderada de V. |
| `MultiHeadAttention` | Instancia `n_head` cabezas independientes, concatena sus salidas y las proyecta de vuelta a `n_embd`. Permite que el modelo atienda a distintos sub-espacios simultáneamente. |
| `FeedForward` | Red MLP de dos capas (expansión 4×) con activación GELU y dropout, aplicada a cada posición de forma independiente. Añade capacidad no-lineal al bloque. |
| `Block` | Combina MHA y FFN con pre-LayerNorm y conexiones residuales: `x = x + MHA(LN(x))`, `x = x + FFN(LN(x))`. Las residuales facilitan el flujo de gradientes. |
| `MiniGPT` | Suma embeddings de tokens y de posición, pasa por `n_layer` bloques, aplica LayerNorm final y proyecta a logits sobre el vocabulario. En inferencia genera token a token. |

### 10.2 Resultados cuantitativos

*Completar con los valores obtenidos al ejecutar las celdas anteriores.*

| Modelo | Parámetros | Train loss | Val loss | Val perplexity | Comentario |
|---|---:|---:|---:|---:|---|
| 1_head   | — | — | — | — | Una sola cabeza, menos diversidad de atención |
| 4_heads  | — | — | — | — | Baseline con 4 cabezas |
| 2_layers | — | — | — | — | Más profundidad, más parámetros |
| block64  | — | — | — | — | Contexto más largo, aprende dependencias más distantes |

### 10.3 Resultados cualitativos

Con corpus mínimo y pocas iteraciones, el modelo aprende rápidamente:

- Estructura de espacios y saltos de línea.
- Combinaciones frecuentes de letras en español (e.g. "que", "el", "la").
- Con más iteraciones, palabras completas del corpus.

No hay coherencia semántica global: el modelo es un LM de nivel carácter con capacidad muy limitada.

### 10.4 Atención

- La parte superior de la matriz de atención es exactamente 0 ✓ (máscara causal respetada).
- La diagonal suele tener peso alto: cada token se presta atención a sí mismo.
- Distintas cabezas tienden a aprender patrones distintos (espacios, vocales, consonantes).

### 10.5 Conclusión

1. **¿Qué aprendió?** Distribución estadística de caracteres condicionada al contexto local. Aprende ortografía básica del español.
2. **Limitaciones:** corpus pequeño, vocabulario de caracteres (no subpalabras), sin memoria más allá de `block_size`.
3. **Diferencias con LLMs modernos:** tokenización subword (BPE), órdenes de magnitud más parámetros, corpus de billones de tokens, RLHF, arquitecturas optimizadas (GQA, RoPE, etc.).
4. **Con más recursos:** aumentar corpus, `block_size`, `n_embd`, `n_layer`; usar tokenizador subword; agregar learning rate scheduler; entrenar en GPU.

## Rúbrica sugerida de corrección

| Ítem | Peso |
|---|---:|
| Implementación correcta de atención causal y tests básicos | 20% |
| Implementación correcta de Multi-Head Attention, FFN, residuales y LayerNorm | 25% |
| Entrenamiento reproducible con curvas y métricas | 15% |
| Evaluación: loss, perplexity, muestras y ablations | 20% |
| Análisis de mapas de atención | 10% |
| Claridad del informe y conclusiones críticas | 10% |

### Extensiones opcionales

- Usar un corpus más grande en español.
- Agregar scheduler de learning rate (cosine decay).
- Guardar y cargar checkpoints.
- Comparar tokenización character-level vs subword.
- Visualizar varias cabezas y capas de atención.

## Referencias

- Vaswani et al. (2017), *Attention Is All You Need*.
- Andrej Karpathy, nanoGPT.
- Material de clase sobre atención, Transformers y modelos autoregresivos.